Looking at Env1 and Env2

In [ ]:
#visuals

import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# --- CONFIGURATION DU STYLE GRAPHIQUE ---
sns.set_theme(style="whitegrid")
plt.rcParams.update(
    {
        "font.size": 18,
        "axes.labelsize": 18,
        "axes.titlesize": 20,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "grid.color": "#FFFFFF",  # Pas de lignes de grille visibles (ou très discrètes)
        "axes.edgecolor": "#FFFFFF", # Supprime les bordures pour un effet flottant
    }
)

# Palette de couleurs exactes de ta seconde image (env1)
WANDB_COLORS = {
    "muon": "#82C9B9",   # Turquoise / Vert d'eau
    "lion": "#D62728",   # Rouge vif
    "adam": "#D162CE",   # Violet / Magenta
    "sgd": "#79BCB4",    # Vert/Bleu pastel (pointillé)
    "adamw": "#D87034",  # Orange / Cuivre
}

def plot_wandb_from_csv(csv_path, metric_substring,nom_fichier):
    if not os.path.exists(csv_path):
        print(f"Erreur : Le fichier {csv_path} est introuvable.")
        return

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    x_col = "iter"
    if x_col not in df.columns:
        print(f"Erreur : La colonne '{x_col}' est introuvable.")
        return

    # Dimensions ajustées
    plt.figure(figsize=(9, 6))
    has_data = False

    for col in df.columns:
        # On ignore les enveloppes Min/Max
        if metric_substring in col and not (
            col.endswith("__MIN") or col.endswith("__MAX")
        ):
            # 1. Extraction du run name
            run_name = col.split(" - ")[0] if " - " in col else col
            
            # 2. Nettoyage strict pour correspondre à la légende de l'image (ex: "env1_muon" -> "muon")
            clean_label = run_name.replace("env1_", "").replace("env2_", "")
            # On retire aussi les suffixes de date ou de chocs potentiels si présents
            clean_label = clean_label.split("_2026")[0].split("_shock")[0]

            # Récupération de la couleur et du style de ligne
            color = WANDB_COLORS.get(clean_label, "#555555")
            linestyle = "--" if "sgd" in clean_label.lower() else "-"
            linewidth = 2.5 if "sgd" in clean_label.lower() else 2.0

            plot_data = df[[x_col, col]].dropna()

            if not plot_data.empty:
                plt.plot(
                    plot_data[x_col],
                    plot_data[col],
                    label=clean_label,
                    color=color,
                    linestyle=linestyle,
                    linewidth=linewidth,
                    alpha=0.9,
                )
                has_data = True

    if not has_data:
        print(f"Aucune donnée trouvée pour '{metric_substring}'.")
        plt.close()
        return

    # --- LOOK & FEEL ASYMÉTRIQUE (W&B STYLE) ---
    plt.xlabel(x_col, loc="right", color="#888888", fontsize=22)
    
    # Formatage de l'axe X (ex: 1000 -> 1k)
    ax = plt.gca()
    ax.xaxis.set_major_formatter(
        plt.FuncFormatter(
            lambda x, loc: f"{int(x/1000)}k" if x >= 1000 else f"{int(x)}"
        )
    )
    
    
    # Aligner le début à 0
    if not df[x_col].empty:
        plt.xlim(0, 4000)

    # Légende parfaitement centrée et espacée en haut
    plt.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, 1.15),
        ncol=3,
        frameon=False,
        fontsize=18,
    )

    # Supprime la ligne du haut, de droite et de gauche. Garde uniquement le bas (X).
    sns.despine(left=True, right=True, top=True, bottom=False)
    
    # Rendre l'axe du bas discret (gris clair)
    ax.spines['bottom'].set_color('#D3D3D3')
    ax.spines['bottom'].set_linewidth(1.5)
    
    # Retirer les petits tics sur les axes pour un look minimaliste
    ax.tick_params(axis='both', which='both', length=0, pad=10)

    plt.tight_layout()

    output_filename = f"wandb_style_output.png"
    plt.savefig(nom_fichier, dpi=300)
    print(f"Graphique sauvegardé : {nom_fichier}")
    plt.show()

# --- EXÉCUTION ---
if __name__ == "__main__":
    mon_fichier_csv = "E2_global_kl.csv"  # Mets le nom de ton fichier CSV ici
    nom_metrique = "global_router/kl_divergence_from_uni" # Change la métrique si besoin
    nom_fichier = "E2_global_kl"
    
    plot_wandb_from_csv(mon_fichier_csv, nom_metrique,nom_fichier)

Erreur : Le fichier E2_global_kl.csv est introuvable.
